# Gridlock 2.0 -- Seatbelt (S3.5): zero-shot baseline, then real two-stage fine-tune (Kaggle T4x2)

**Local CPU baseline to beat:** MobileNetV3-small, 224px, windshield-crop binary classifier
(seatbelt / no_seatbelt), 3-attempt cap reached locally (4 GB GPU): **best val F1 = 0.678,
acc = 0.766** (`checkpoints/seatbelt/v2/`). Honest label semantics: a windshield crop is
labeled `no_seatbelt` if no annotated belt box lies inside it -- supervision, not perfect GT
(night/tint/glare/rear-occupant are real failure modes per S3.5).

**Same workflow as detection:** zero-shot test first, then fine-tune to beat it -- on GPU this
time, and implementing the **actual documented two-stage architecture** (S3.5: *"YOLOv11/
windshield detector -> driver crop -> CNN/CNN-SVM belt classifier"*), not just a classifier
fed ground-truth crops.

- **Part A -- zero-shot:** SAM-3 (native API, text-prompted) classifies each windshield crop as
  belt/no-belt directly, no training at all. This is the number to beat, same role as the
  zero-shot RF-DETR baseline before the detection fine-tune.
- **Part B -- fine-tune (real two-stage):**
  1. **YOLOv11** windshield detector (Ultralytics, **AGPL-3.0 -> benchmark-only**, matches the
     documented model name exactly -- same license flag pattern as the detection notebook's
     YOLOv11-l comparator).
  2. **MobileNetV3-small** belt classifier, same recipe as the local run but full GPU capacity
     (more epochs, higher res, full dataset, no 4 GB ceiling).
  3. **End-to-end eval:** Stage-1 detector's own windshield boxes -> Stage-2 classifier -- the
     true two-stage number, not classifier-on-GT-crop.


## 0. Install dependencies

In [ ]:
import numpy
_PINNED_NUMPY = numpy.__version__
print('numpy before install:', _PINNED_NUMPY)

!pip -q install ultralytics pycocotools "numpy=={_PINNED_NUMPY}" 2>/dev/null

# SAM-3: native package (matches the validated reference-notebook API), not the HF wrapper.
!git clone -q https://github.com/facebookresearch/sam3.git /kaggle/working/sam3_repo
!pip -q install -e "/kaggle/working/sam3_repo[notebooks]" "numpy=={_PINNED_NUMPY}" 2>/dev/null
print('installed (numpy pinned to', _PINNED_NUMPY, ')')


### Restart the kernel now -- before running anything below
Same numpy/torchvision ABI-mismatch reason as the other two notebooks. **Kaggle:** session menu
-> **Restart Session**, then continue from the next cell. Do not re-run the install cell.

In [ ]:
import torch, time, json as _json
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {round(p.total_memory/1024**3,1)} GB')


## 1. HF token (gated repo -- never hard-code)
Add via Kaggle: **Add-ons -> Secrets -> `HF_TOKEN`**.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    if os.environ.get('HF_TOKEN'):
        print('HF_TOKEN loaded from environment')
    else:
        print('WARNING: no HF_TOKEN found -- add it via Kaggle Secrets before running Part A.', e)


## 2. Config -- EDIT `SB_ROOT`
Upload the `seat_belt and mobile.v2i.yolov8-obb` dataset (the same one used for the local run)
and point `SB_ROOT` at the folder containing `train/` and `valid/` (each with `images/`,
`labels/`).

In [ ]:
from pathlib import Path

# >>> EDIT THIS to your uploaded dataset path <<<
SB_ROOT = Path('/kaggle/input/seat-belt-detection/seat_belt and mobile.v2i.yolov8-obb')

WORK = Path('/kaggle/working')
YOLO_DIR = WORK / 'windshield_yolo'
CKPT_DIR = WORK / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# label classes in the source OBB dataset (matches src/train/train_seatbelt.py)
MOBILE, BELT, WINDSHIELD = 0, 1, 2
PSEUDO_CONF = None  # unused here, placeholder removed below

assert SB_ROOT.exists(), f'SB_ROOT not found: {SB_ROOT} (check /kaggle/input)'
for sp in ('train', 'valid'):
    print(sp, '->', (SB_ROOT / sp / 'images').exists(), (SB_ROOT / sp / 'labels').exists())


## 3. Parse OBB labels -> windshield crops + binary belt labels
Same logic as the local `train_seatbelt.py`: crop each **windshield** box (5% margin); label
= `no_seatbelt` (1, the violation) if no **belt** box lies >50% inside it, else `seatbelt` (0).
`mobile` instances are counted but out of scope (not a mandated violation).

In [ ]:
import cv2
import numpy as np
from PIL import Image

def obb_to_xyxy(coords):
    xs = coords[0::2]; ys = coords[1::2]
    return (min(xs), min(ys), max(xs), max(ys))

def parse_obb_label_file(path):
    out = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        toks = line.split()
        if len(toks) != 9:
            continue
        cls = int(float(toks[0]))
        coords = [float(t) for t in toks[1:]]
        out.append({'class_id': cls, 'aabb_xyxy': obb_to_xyxy(coords)})
    return out

def _contain_frac(inner, outer):
    ix1, iy1, ix2, iy2 = inner; ox1, oy1, ox2, oy2 = outer
    x1, y1, x2, y2 = max(ix1, ox1), max(iy1, oy1), min(ix2, ox2), min(iy2, oy2)
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a = (ix2 - ix1) * (iy2 - iy1)
    return inter / a if a > 0 else 0.0

def collect_split(split, margin=0.05):
    """Returns list of dicts: img_path, windshield_px(x1,y1,x2,y2), label, crop(PIL)."""
    img_dir = SB_ROOT / split / 'images'
    lbl_dir = SB_ROOT / split / 'labels'
    items, mobiles = [], 0
    if not img_dir.exists():
        return items, mobiles
    for img_path in sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png')):
        lbl = lbl_dir / (img_path.stem + '.txt')
        if not lbl.exists():
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        recs = parse_obb_label_file(lbl)
        winds = [r['aabb_xyxy'] for r in recs if r['class_id'] == WINDSHIELD]
        belts = [r['aabb_xyxy'] for r in recs if r['class_id'] == BELT]
        mobiles += sum(1 for r in recs if r['class_id'] == MOBILE)
        for wd in winds:
            x1, y1, x2, y2 = wd
            bw, bh = (x2 - x1) * w, (y2 - y1) * h
            cx1 = int(max(0, x1 * w - margin * bw)); cy1 = int(max(0, y1 * h - margin * bh))
            cx2 = int(min(w, x2 * w + margin * bw)); cy2 = int(min(h, y2 * h + margin * bh))
            if cx2 - cx1 < 8 or cy2 - cy1 < 8:
                continue
            has_belt = any(_contain_frac(b, wd) > 0.5 for b in belts)
            label = 0 if has_belt else 1   # 1 = no_seatbelt (violation)
            crop = Image.fromarray(img[cy1:cy2, cx1:cx2][:, :, ::-1])
            items.append({'img_path': img_path, 'windshield_px': (cx1, cy1, cx2, cy2),
                          'label': label, 'crop': crop})
    return items, mobiles

train_items, train_mobiles = collect_split('train')
valid_items, valid_mobiles = collect_split('valid')
from collections import Counter
print('train:', len(train_items), Counter(it['label'] for it in train_items), 'mobile(out-of-scope)=', train_mobiles)
print('valid:', len(valid_items), Counter(it['label'] for it in valid_items), 'mobile(out-of-scope)=', valid_mobiles)


## 4. Load SAM-3 (native API)

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

sam3_model = build_sam3_image_model(
    bpe_path='/kaggle/working/sam3_repo/sam3/assets/bpe_simple_vocab_16e6.txt.gz')
sam3_processor = Sam3Processor(sam3_model, confidence_threshold=0.2)
print('SAM-3 loaded')


## 5. Part A -- zero-shot: SAM-3 classifies belt / no-belt directly (no training)
For each windshield crop, run both prompt sets and take the side with the higher max score.
If neither concept fires at all, the crop is **not scored** (coverage gap) rather than guessed --
honest reporting, same principle as the Track-B foundation-model tests
(`docs/final/07_trackB_foundation_models_lightning.md`).

In [ ]:
SEATBELT_PROMPTS = {
    0: ['driver wearing a seat belt', 'person in car wearing seatbelt', 'seat belt across the chest'],
    1: ['car driver without seatbelt', 'driver not wearing seat belt', 'person in car not wearing seatbelt'],
}

def sam3_best_score(pil_img, phrases):
    best = 0.0
    state = sam3_processor.set_image(pil_img)
    for phrase in phrases:
        state = sam3_processor.set_text_prompt(state=state, prompt=phrase)
        scores = state['scores'].to(torch.float32).cpu().numpy()
        if scores.size:
            best = max(best, float(scores.max()))
    return best

t0 = time.time()
zs_preds, zs_labels = [], []
n_unscored = 0
for i, it in enumerate(valid_items, 1):
    score_belt = sam3_best_score(it['crop'], SEATBELT_PROMPTS[0])
    score_nobelt = sam3_best_score(it['crop'], SEATBELT_PROMPTS[1])
    if score_belt == 0.0 and score_nobelt == 0.0:
        n_unscored += 1
        continue
    zs_preds.append(1 if score_nobelt >= score_belt else 0)
    zs_labels.append(it['label'])
    if i % 50 == 0:
        print(f'  zero-shot {i}/{len(valid_items)}  ({time.time()-t0:.0f}s)')

print(f'\nzero-shot done in {time.time()-t0:.0f}s | scored {len(zs_preds)}/{len(valid_items)} '
      f'(unscored/no-fire: {n_unscored})')


In [ ]:
def prf1(preds, labels, positive=1):
    tp = sum(1 for p, y in zip(preds, labels) if p == positive and y == positive)
    fp = sum(1 for p, y in zip(preds, labels) if p == positive and y != positive)
    fn = sum(1 for p, y in zip(preds, labels) if p != positive and y == positive)
    tn = sum(1 for p, y in zip(preds, labels) if p != positive and y != positive)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    acc = (tp + tn) / max(len(preds), 1)
    return {'accuracy': round(acc, 4), 'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4)}

zs_metrics = prf1(zs_preds, zs_labels)
print('ZERO-SHOT SAM-3 (no_seatbelt = positive class):', zs_metrics)
print('local CPU MobileNetV3-small baseline (for comparison): acc=0.766 f1=0.678')


## 6. Part B, Stage 1 -- YOLOv11 windshield detector fine-tune
Matches S3.5's documented model name exactly. **Ultralytics YOLO = AGPL-3.0** -> internal
benchmark/pipeline-stage only, same license flag as the detection notebook's YOLOv11-l
comparator (S12 licensing hygiene).

In [ ]:
import shutil

def build_windshield_yolo(split, roboflow_name):
    img_dir = SB_ROOT / split / 'images'
    lbl_dir = SB_ROOT / split / 'labels'
    odir = YOLO_DIR / roboflow_name
    (odir / 'images').mkdir(parents=True, exist_ok=True)
    (odir / 'labels').mkdir(parents=True, exist_ok=True)
    n = 0
    for img_path in sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png')):
        lbl = lbl_dir / (img_path.stem + '.txt')
        if not lbl.exists():
            continue
        recs = parse_obb_label_file(lbl)
        winds = [r['aabb_xyxy'] for r in recs if r['class_id'] == WINDSHIELD]
        if not winds:
            continue
        dst_img = odir / 'images' / img_path.name
        if not dst_img.exists():
            try:
                os.symlink(img_path, dst_img)
            except Exception:
                shutil.copy2(img_path, dst_img)
        lines = []
        for x1, y1, x2, y2 in winds:
            xc, yc, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
            lines.append(f'0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        (odir / 'labels' / (img_path.stem + '.txt')).write_text('\n'.join(lines) + '\n')
        n += 1
    return n

import os
n_train = build_windshield_yolo('train', 'train')
n_valid = build_windshield_yolo('valid', 'valid')
(YOLO_DIR / 'data.yaml').write_text(
    f"train: {(YOLO_DIR/'train'/'images').as_posix()}\n"
    f"val: {(YOLO_DIR/'valid'/'images').as_posix()}\n"
    f"nc: 1\nnames:\n  0: windshield\n")
print(f'windshield-only YOLO set: train={n_train} valid={n_valid} -> {YOLO_DIR/"data.yaml"}')


In [ ]:
from ultralytics import YOLO

windshield_det = YOLO('yolo11n.pt')   # single easy class -> nano is enough
wd_res = windshield_det.train(
    data=str(YOLO_DIR / 'data.yaml'), epochs=60, imgsz=640, batch=16,
    device=0, workers=2, patience=20, amp=True,
    project=str(WORK / 'yolo11n_windshield'), name='ft', exist_ok=True, plots=False, verbose=True)
print('windshield detector done')


## 7. Part B, Stage 2 -- belt classifier fine-tune (full GPU capacity)
Same MobileNetV3-small recipe as the local run, but no 4 GB ceiling: higher resolution, more
epochs, full dataset, no frozen backbone.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

INPUT_SIZE = 224
EPOCHS = 40
BATCH = 64
LR = 1e-4

tfm_train = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(),          # belt presence is flip-invariant -- safe here
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
tfm_eval = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class CropDataset(Dataset):
    def __init__(self, items, tfm):
        self.items = items; self.tfm = tfm
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        return self.tfm(it['crop']), it['label']

train_ds = CropDataset(train_items, tfm_train)
val_ds = CropDataset(valid_items, tfm_eval)

counts = [0, 0]
for it in train_items:
    counts[it['label']] += 1
total = sum(counts)
weights = torch.tensor([total / (2 * max(c, 1)) for c in counts], dtype=torch.float32)
print('train class counts (seatbelt, no_seatbelt):', counts)

w = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
belt_model = models.mobilenet_v3_small(weights=w)
in_f = belt_model.classifier[-1].in_features
belt_model.classifier[-1] = nn.Linear(in_f, 2)
belt_model.to('cuda')

opt = torch.optim.Adam(belt_model.parameters(), lr=LR, weight_decay=1e-4)
crit = nn.CrossEntropyLoss(weight=weights.to('cuda'))
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2)

def evaluate(model, dl):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to('cuda')
            p = model(xb).argmax(1).cpu().tolist()
            preds.extend(p); labels.extend(yb.tolist())
    return prf1(preds, labels), preds, labels

best_f1 = -1.0
ckpt_path = CKPT_DIR / 'belt_classifier.pt'
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    belt_model.train()
    run_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to('cuda'), yb.to('cuda')
        opt.zero_grad()
        loss = crit(belt_model(xb), yb)
        loss.backward(); opt.step()
        run_loss += float(loss) * len(xb)
    vm, _, _ = evaluate(belt_model, val_dl)
    if vm['f1'] >= best_f1:
        best_f1 = vm['f1']
        torch.save({'state_dict': belt_model.state_dict(), 'input': INPUT_SIZE}, ckpt_path)
    if ep % 5 == 0 or ep == EPOCHS:
        print(f'epoch {ep}/{EPOCHS} loss={run_loss/len(train_ds):.4f} val_f1={vm["f1"]} acc={vm["accuracy"]}')

print(f'\nbelt classifier done in {(time.time()-t0)/60:.1f} min | best val F1 (classifier-only, GT crops) = {best_f1}')
print('local CPU baseline f1=0.678 -- classifier-only comparison is apples-to-apples with that number.')


## 8. End-to-end two-stage eval -- detector's own boxes -> classifier
The actual S3.5 architecture number: Stage-1 windshield detector finds the crop (no GT boxes
used here), Stage-2 classifier reads it. IoU-matched to GT windshield boxes to assign the
correct label per detection.

In [ ]:
def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    aa = (ax2 - ax1) * (ay2 - ay1); ab = (bx2 - bx1) * (by2 - by1)
    return inter / (aa + ab - inter) if (aa + ab - inter) > 0 else 0.0

belt_model.eval()
ckpt = torch.load(ckpt_path, map_location='cuda')
belt_model.load_state_dict(ckpt['state_dict'])

# group GT windshield items by source image for IoU matching
from collections import defaultdict
gt_by_img = defaultdict(list)
for it in valid_items:
    gt_by_img[str(it['img_path'])].append(it)

e2e_preds, e2e_labels = [], []
n_det_total = n_matched = 0
for img_path in sorted(set(str(it['img_path']) for it in valid_items)):
    res = windshield_det.predict(img_path, conf=0.3, verbose=False)[0]
    img = cv2.imread(img_path)
    for box in res.boxes.xyxy.cpu().numpy():
        n_det_total += 1
        x1, y1, x2, y2 = [int(v) for v in box]
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        best_gt, best_iou = None, 0.0
        for it in gt_by_img[img_path]:
            iou = iou_xyxy((x1, y1, x2, y2), it['windshield_px'])
            if iou > best_iou:
                best_iou, best_gt = iou, it
        if best_gt is None or best_iou < 0.5:
            continue   # detection doesn't correspond to a labeled windshield -- skip, don't guess
        n_matched += 1
        crop = Image.fromarray(img[y1:y2, x1:x2][:, :, ::-1])
        xb = tfm_eval(crop).unsqueeze(0).to('cuda')
        with torch.no_grad():
            pred = belt_model(xb).argmax(1).item()
        e2e_preds.append(pred)
        e2e_labels.append(best_gt['label'])

e2e_metrics = prf1(e2e_preds, e2e_labels) if e2e_preds else {'note': 'no IoU-matched detections'}
print(f'detector boxes: {n_det_total} | IoU>=0.5 matched to GT windshield: {n_matched}')
print('END-TO-END two-stage (detector boxes -> classifier):', e2e_metrics)


## How to read this
| Stage | Metric (no_seatbelt F1) |
|---|---|
| Local CPU baseline (MobileNetV3, GT crops, 4 GB cap) | 0.678 |
| **Zero-shot SAM-3** (no training, GT crops) | see Part A output |
| **Fine-tuned classifier-only** (Kaggle GPU, GT crops -- apples-to-apples vs the 0.678 baseline) | see Part B Stage-2 output |
| **Full two-stage** (own detector boxes -> classifier -- the real S3.5 number) | see Part B end-to-end output |

- If classifier-only beats 0.678, the extra GPU capacity (res 224, 40 epochs, full data, no
  frozen backbone) helped on its own -- expected, since the local run hit a 3-attempt cap on
  4 GB CPU, not a data/architecture ceiling.
- The **end-to-end** number is usually a bit lower than classifier-only -- it now also pays for
  Stage-1 detector recall/localization error, which the GT-crop number never had to.
- **Honest scope still applies:** night/tint/glare/rear-occupant are still failure modes (S3.5);
  label semantics are still "no annotated belt box" not perfect ground truth.
